# 随机抽取并核验 PyG Data

本 notebook 从处理完成的 `.pt` 文件中可复现地随机抽样，并将节点、空间边、ligand 切割连接和对象属性整理成表格。

节点表中的 14 个 `x` 特征会按照 `PhiSSeparator_SE3TD_3D.py` 的规则反向归一化。请注意：原流程对角度特征限制到 `[0, 1]`，对其余归一化特征限制到 `[-6, 6]`；落在这些边界上的值可能已经丢失原始幅度，因此只能恢复到相应边界值。全局虚拟节点不参与反向归一化，其特征显示为缺失值。

`ligand_connections` 会展示两个片段节点、各自的规范原子编号、元素身份和保存的原子对距离。当前 `.pt` 不保存连接原子的独立坐标，因此这里只校验距离有限且非负，不能重新计算距离。距离单位为 Å。

> 安全提示：`torch.load` 会反序列化 Python 对象，只应加载本项目自行生成且来源可信的 `.pt` 文件。

In [ ]:
from pathlib import Path
import json
import random
import re

import numpy as np
import pandas as pd
import torch
from rdkit import Chem
from IPython.display import Markdown, display

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 180)

# ------------------------------
# 用户配置
# ------------------------------
N_SAMPLES = 3
RANDOM_SEED = 42
ROUND_DIGITS = 6
EXPORT_CSV = False

FEATURE_COLUMNS = [
    "avg_ecc", "ecc_range", "avg_logp", "tpsa", "avg_charge",
    "HBA", "HBD", "Aromatic", "Hydrophobe", "PosIonizable", "NegIonizable",
    "max_dist_3d", "max_angle_3d", "max_plane_angle_3d",
]
ANGLE_FEATURES = {"max_angle_3d": 180.0, "max_plane_angle_3d": 180.0}
COUNT_FEATURES = {
    "HBA", "HBD", "Aromatic", "Hydrophobe", "PosIonizable", "NegIonizable"
}
NODE_TYPE_NAMES = {0: "ligand", 1: "pocket", 2: "global"}
EDGE_TYPE_NAMES = {0: "LL", 1: "LP", 2: "PP", 3: "DL"}


def find_project_root(start: Path) -> Path:
    """允许从项目根目录、ChemCut 目录或其子目录启动 notebook。"""
    start = start.resolve()
    for candidate in (start, *start.parents):
        marker = candidate / "ChemCut" / "PhiSSeparator_SE3TD_3D.py"
        if marker.is_file():
            return candidate
    raise FileNotFoundError("无法定位 PhiStone_2.0 项目根目录，请手动设置 PROJECT_ROOT。")


PROJECT_ROOT = find_project_root(Path.cwd())
OUTPUT_ROOT = (
    PROJECT_ROOT / "Datas" / "Processed_datas" / "SE3TD_HiQBind_PDBbind_128_Pygdata"
)
# 若需检查离线 128 维回填后的数据，只需把 DATA_DIR 改为对应目录。
DATA_DIR = OUTPUT_ROOT / "pyg_pending"
SHARED_DATA_DIR = (
    PROJECT_ROOT / "Datas" / "Processed_datas" / "SE3TD_128_Shared_data"
)
STATS_PATH = SHARED_DATA_DIR / "normalization_stats.json"
EXPORT_DIR = PROJECT_ROOT / "ChemCut" / "pyg_sample_review"

print(f"项目根目录: {PROJECT_ROOT}")
print(f"PyG 数据目录: {DATA_DIR}")
print(f"归一化统计: {STATS_PATH}")

## 加载、反向归一化与表格构建函数

In [ ]:
def load_normalization_params(stats_path: Path) -> dict:
    if not stats_path.is_file():
        raise FileNotFoundError(f"未找到归一化统计文件: {stats_path}")
    with stats_path.open("r", encoding="utf-8") as file:
        raw_stats = json.load(file)

    missing = set(FEATURE_COLUMNS) - set(raw_stats)
    if missing:
        raise ValueError(f"归一化统计缺少特征: {sorted(missing)}")

    params = {}
    for name in FEATURE_COLUMNS:
        stats = raw_stats[name]
        if not {"count", "mean", "M2"} <= set(stats):
            raise ValueError(f"特征 {name} 的统计格式不完整")
        count = float(stats["count"])
        mean = float(stats["mean"])
        m2 = float(stats["M2"])
        if count <= 0 or not np.isfinite([count, mean, m2]).all() or m2 < 0:
            raise ValueError(f"特征 {name} 的统计值非法: {stats}")
        std = float(np.sqrt(m2 / count)) if count >= 2 else 0.0
        params[name] = {"count": count, "mean": mean, "std": std}
    return params


def discover_and_sample(data_dir: Path, sample_count: int, seed: int) -> list[Path]:
    if sample_count <= 0:
        raise ValueError("N_SAMPLES 必须为正整数")
    if not data_dir.is_dir():
        raise FileNotFoundError(f"PyG 数据目录不存在: {data_dir}")
    files = sorted(data_dir.rglob("*.pt"), key=lambda path: path.as_posix())
    if not files:
        raise FileNotFoundError(f"目录中未找到 .pt 文件: {data_dir}")
    actual_count = min(sample_count, len(files))
    if actual_count < sample_count:
        print(f"仅找到 {len(files)} 个 .pt 文件，将全部载入。")
    selected = random.Random(seed).sample(files, actual_count)
    return sorted(selected, key=lambda path: path.as_posix())


def trusted_torch_load(path: Path):
    """显式允许加载本项目可信的 PyG Data；兼容不支持 weights_only 的旧版 PyTorch。"""
    try:
        return torch.load(path, map_location="cpu", weights_only=False)
    except TypeError:
        return torch.load(path, map_location="cpu")


def to_numpy(value):
    if isinstance(value, torch.Tensor):
        return value.detach().cpu().numpy()
    return np.asarray(value)


def tensor_attr(data, name: str):
    value = getattr(data, name, None)
    return value if isinstance(value, torch.Tensor) else None


def node_roles(data) -> list[str]:
    node_type = tensor_attr(data, "node_type")
    num_nodes = int(data.x.shape[0])
    if node_type is None or node_type.ndim != 2 or node_type.shape[0] != num_nodes:
        return ["unknown"] * num_nodes
    roles = []
    for row in to_numpy(node_type):
        index = int(np.argmax(row)) if row.size else -1
        roles.append(NODE_TYPE_NAMES.get(index, "unknown") if np.max(row) > 0 else "unknown")
    return roles


def inverse_feature(name: str, normalized_value: float, params: dict) -> tuple[float, bool]:
    """返回反归一化值，以及该值是否位于原流程的截断边界。"""
    value = float(normalized_value)
    stats = params[name]
    if name in ANGLE_FEATURES:
        restored = value * ANGLE_FEATURES[name]
        at_boundary = bool(np.isclose(value, 0.0) or np.isclose(value, 1.0))
    elif name in COUNT_FEATURES:
        restored = value * max(stats["std"], 1.0) + stats["mean"]
        at_boundary = bool(np.isclose(abs(value), 6.0))
    else:
        safe_std = stats["std"] if stats["std"] > 1e-5 else 1.0
        restored = value * safe_std + stats["mean"]
        at_boundary = bool(np.isclose(abs(value), 6.0))
    return float(restored), at_boundary


def scalar_or_none(value):
    if isinstance(value, torch.Tensor) and value.numel() == 1:
        return value.detach().cpu().item()
    return value


def safe_filename(value) -> str:
    return re.sub(r"[^A-Za-z0-9_.-]+", "_", str(value)).strip("_") or "sample"

In [ ]:
def build_node_rows(data, path: Path, params: dict) -> list[dict]:
    x = tensor_attr(data, "x")
    if x is None or x.ndim != 2 or x.shape[1] < len(FEATURE_COLUMNS):
        shape = None if x is None else tuple(x.shape)
        raise ValueError(f"{path.name}: x 形状不足以容纳 14 个特征，实际为 {shape}")

    x_np = to_numpy(x[:, :len(FEATURE_COLUMNS)])
    roles = node_roles(data)
    num_nodes = x_np.shape[0]
    smiles = list(getattr(data, "fragment_smiles", []))
    metal_mask = tensor_attr(data, "metal_mask")
    hac = tensor_attr(data, "hac")
    ring_count = tensor_attr(data, "ring_count")
    pos = tensor_attr(data, "pos")
    ref_coords = tensor_attr(data, "ref_coords")

    metal_np = to_numpy(metal_mask) if metal_mask is not None else None
    hac_np = to_numpy(hac) if hac is not None else None
    ring_np = to_numpy(ring_count) if ring_count is not None else None
    pos_np = to_numpy(pos) if pos is not None else None
    ref_np = to_numpy(ref_coords) if ref_coords is not None else None

    sample_id = str(getattr(data, "sample_id", path.stem))
    rows = []
    for node_index in range(num_nodes):
        is_global = roles[node_index] == "global" or (
            node_index < len(smiles) and smiles[node_index] == "<GLOBAL>"
        )
        row = {
            "sample_id": sample_id,
            "file": str(path.relative_to(DATA_DIR)),
            "node_index": node_index,
            "node_role": roles[node_index],
            "fragment_smiles": smiles[node_index] if node_index < len(smiles) else None,
            "is_metal": bool(metal_np[node_index]) if metal_np is not None and node_index < len(metal_np) else None,
            "hac": int(hac_np[node_index]) if hac_np is not None and node_index < len(hac_np) else None,
            "ring_count": int(ring_np[node_index]) if ring_np is not None and node_index < len(ring_np) else None,
        }

        boundary_features = []
        for column_index, name in enumerate(FEATURE_COLUMNS):
            if is_global:
                row[name] = np.nan
                continue
            restored, at_boundary = inverse_feature(name, x_np[node_index, column_index], params)
            row[name] = round(restored, ROUND_DIGITS)
            if at_boundary:
                boundary_features.append(name)
        row["inverse_boundary_features"] = ", ".join(boundary_features)

        if pos_np is not None and pos_np.ndim == 2 and pos_np.shape[1] >= 3:
            row.update({
                "pos_x": round(float(pos_np[node_index, 0]), ROUND_DIGITS),
                "pos_y": round(float(pos_np[node_index, 1]), ROUND_DIGITS),
                "pos_z": round(float(pos_np[node_index, 2]), ROUND_DIGITS),
            })
        if ref_np is not None and ref_np.ndim == 3 and ref_np.shape[1:] == (3, 3):
            for ref_index in range(3):
                for axis_index, axis in enumerate("xyz"):
                    row[f"ref{ref_index}_{axis}"] = round(
                        float(ref_np[node_index, ref_index, axis_index]), ROUND_DIGITS
                    )
        rows.append(row)
    return rows


def edge_type_from_row(edge_attr_row) -> str:
    values = np.asarray(edge_attr_row)
    if values.ndim != 1 or values.size == 0:
        return "unknown"
    return EDGE_TYPE_NAMES.get(int(np.argmax(values)), "unknown")


def build_edge_rows(data, path: Path) -> list[dict]:
    edge_index = tensor_attr(data, "edge_index")
    edge_attr = tensor_attr(data, "edge_attr")
    if edge_index is None or edge_index.ndim != 2 or edge_index.shape[0] != 2:
        return []
    edge_index_np = to_numpy(edge_index)
    edge_attr_np = to_numpy(edge_attr) if edge_attr is not None else None
    roles = node_roles(data)
    sample_id = str(getattr(data, "sample_id", path.stem))

    unique_edges = {}
    for edge_number in range(edge_index_np.shape[1]):
        source = int(edge_index_np[0, edge_number])
        target = int(edge_index_np[1, edge_number])
        key = (min(source, target), max(source, target))
        edge_type = (
            edge_type_from_row(edge_attr_np[edge_number])
            if edge_attr_np is not None and edge_number < len(edge_attr_np)
            else "unknown"
        )
        if key not in unique_edges:
            unique_edges[key] = {
                "sample_id": sample_id,
                "file": str(path.relative_to(DATA_DIR)),
                "node_u": key[0],
                "node_v": key[1],
                "role_u": roles[key[0]] if 0 <= key[0] < len(roles) else "out_of_range",
                "role_v": roles[key[1]] if 0 <= key[1] < len(roles) else "out_of_range",
                "edge_type": edge_type,
                "directed_occurrences": 0,
            }
        unique_edges[key]["directed_occurrences"] += 1
        if unique_edges[key]["edge_type"] != edge_type:
            unique_edges[key]["edge_type"] = "inconsistent"
    return list(unique_edges.values())


def fragment_molecules(data) -> tuple[list[str], list]:
    x = tensor_attr(data, "x")
    num_nodes = int(x.shape[0]) if x is not None and x.ndim >= 1 else 0
    raw_smiles = getattr(data, "fragment_smiles", [])
    smiles = list(raw_smiles) if isinstance(raw_smiles, (list, tuple)) else []
    molecules = []
    for node_index in range(num_nodes):
        value = str(smiles[node_index]) if node_index < len(smiles) else ""
        molecules.append(
            None if not value or value == "<GLOBAL>" else Chem.MolFromSmiles(value)
        )
    return smiles, molecules


def build_connection_rows(data, path: Path) -> list[dict]:
    connection_edges = tensor_attr(data, "connection_edge_index")
    connection_atoms = tensor_attr(data, "connection_atom_ids")
    connection_distances = tensor_attr(data, "connection_distance")
    if (
        connection_edges is None
        or connection_edges.ndim != 2
        or connection_edges.shape[0] != 2
    ):
        return []

    num_connections = int(connection_edges.shape[1])
    edges_np = to_numpy(connection_edges)
    atoms_np = (
        to_numpy(connection_atoms)
        if connection_atoms is not None
        and tuple(connection_atoms.shape) == (num_connections, 2)
        else None
    )
    distances_np = (
        to_numpy(connection_distances)
        if connection_distances is not None
        and tuple(connection_distances.shape) == (num_connections,)
        else None
    )
    roles = node_roles(data)
    smiles, molecules = fragment_molecules(data)
    sample_id = str(getattr(data, "sample_id", path.stem))

    node_pairs = [
        tuple(sorted((int(edges_np[0, index]), int(edges_np[1, index]))))
        for index in range(num_connections)
    ]
    pair_counts = {}
    for node_pair in node_pairs:
        pair_counts[node_pair] = pair_counts.get(node_pair, 0) + 1

    def atom_symbol(node_index, atom_index):
        if (
            atom_index is None
            or not 0 <= node_index < len(molecules)
            or molecules[node_index] is None
            or not 0 <= atom_index < molecules[node_index].GetNumAtoms()
        ):
            return None
        return molecules[node_index].GetAtomWithIdx(atom_index).GetSymbol()

    rows = []
    for connection_index in range(num_connections):
        node_a = int(edges_np[0, connection_index])
        node_b = int(edges_np[1, connection_index])
        atom_a = int(atoms_np[connection_index, 0]) if atoms_np is not None else None
        atom_b = int(atoms_np[connection_index, 1]) if atoms_np is not None else None

        distance = (
            float(distances_np[connection_index])
            if distances_np is not None else None
        )
        rows.append({
            "sample_id": sample_id,
            "file": str(path.relative_to(DATA_DIR)),
            "connection_index": connection_index,
            "node_a": node_a,
            "role_a": roles[node_a] if 0 <= node_a < len(roles) else "out_of_range",
            "fragment_smiles_a": smiles[node_a] if 0 <= node_a < len(smiles) else None,
            "canonical_atom_id_a": atom_a,
            "element_a": atom_symbol(node_a, atom_a),
            "node_b": node_b,
            "role_b": roles[node_b] if 0 <= node_b < len(roles) else "out_of_range",
            "fragment_smiles_b": smiles[node_b] if 0 <= node_b < len(smiles) else None,
            "canonical_atom_id_b": atom_b,
            "element_b": atom_symbol(node_b, atom_b),
            "node_pair_occurrences": pair_counts[node_pairs[connection_index]],
            "distance_angstrom": (
                round(distance, ROUND_DIGITS) if distance is not None else None
            ),
        })
    return rows


def summarize_attribute(value) -> tuple[str, str, str]:
    if isinstance(value, torch.Tensor):
        flat = value.detach().cpu().reshape(-1)
        preview = flat[:8].tolist()
        return str(tuple(value.shape)), str(value.dtype), repr(preview)
    if isinstance(value, (list, tuple)):
        return f"len={len(value)}", type(value).__name__, repr(value[:5])
    return "scalar", type(value).__name__, repr(value)


def build_embedding_rows(data, path: Path) -> list[dict]:
    frag_embeds = tensor_attr(data, "frag_embeds")
    if frag_embeds is None:
        return []
    if frag_embeds.ndim != 2 or frag_embeds.shape[0] != data.x.shape[0]:
        raise ValueError(
            f"{path.name}: frag_embeds 应为 [num_nodes, embedding_dim]，"
            f"实际为 {tuple(frag_embeds.shape)}"
        )
    embeds_np = to_numpy(frag_embeds)
    roles = node_roles(data)
    sample_id = str(getattr(data, "sample_id", path.stem))
    rows = []
    for node_index, vector in enumerate(embeds_np):
        row = {
            "sample_id": sample_id,
            "file": str(path.relative_to(DATA_DIR)),
            "node_index": node_index,
            "node_role": roles[node_index],
        }
        row.update({
            f"embedding_{dimension:03d}": round(float(value), ROUND_DIGITS)
            for dimension, value in enumerate(vector)
        })
        rows.append(row)
    return rows


def build_attribute_rows(data, path: Path) -> list[dict]:
    sample_id = str(getattr(data, "sample_id", path.stem))
    keys = sorted(list(data.keys()))
    rows = []
    for key in keys:
        shape, dtype, preview = summarize_attribute(getattr(data, key))
        rows.append({
            "sample_id": sample_id,
            "file": str(path.relative_to(DATA_DIR)),
            "attribute": key,
            "shape_or_length": shape,
            "dtype": dtype,
            "preview": preview,
        })
    return rows

In [ ]:
def finite_tensor(data, name: str) -> bool:
    value = tensor_attr(data, name)
    if value is None or not (value.is_floating_point() or value.is_complex()):
        return True
    return bool(torch.isfinite(value).all().item())


def build_summary_row(data, path: Path) -> dict:
    x = tensor_attr(data, "x")
    edge_index = tensor_attr(data, "edge_index")
    edge_attr = tensor_attr(data, "edge_attr")
    roles = node_roles(data) if x is not None else []
    edge_counts = {name: 0 for name in EDGE_TYPE_NAMES.values()}
    if edge_attr is not None and edge_attr.ndim == 2:
        for row in to_numpy(edge_attr):
            name = edge_type_from_row(row)
            edge_counts[name] = edge_counts.get(name, 0) + 1
    connection_edges = tensor_attr(data, "connection_edge_index")
    connection_distances = tensor_attr(data, "connection_distance")
    frag_embeds = tensor_attr(data, "frag_embeds")
    distance_values = (
        to_numpy(connection_distances)
        if connection_distances is not None and connection_distances.ndim == 1 else None
    )
    has_distances = distance_values is not None and distance_values.size > 0
    connection_count = (
        int(connection_edges.shape[1])
        if connection_edges is not None
        and connection_edges.ndim == 2
        and connection_edges.shape[0] == 2
        else None
    )
    node_pair_counts = {}
    if connection_count is not None:
        for node_a, node_b in connection_edges.t().tolist():
            node_pair = tuple(sorted((int(node_a), int(node_b))))
            node_pair_counts[node_pair] = node_pair_counts.get(node_pair, 0) + 1
    return {
        "sample_id": str(getattr(data, "sample_id", path.stem)),
        "pdb_id": str(getattr(data, "pdb_id", "")),
        "source": str(getattr(data, "source", "")),
        "file": str(path.relative_to(DATA_DIR)),
        "num_nodes": int(x.shape[0]) if x is not None and x.ndim >= 1 else None,
        "ligand_nodes": roles.count("ligand"),
        "pocket_nodes": roles.count("pocket"),
        "global_nodes": roles.count("global"),
        "x_dim": int(x.shape[1]) if x is not None and x.ndim == 2 else None,
        "frag_embed_dim": (
            int(frag_embeds.shape[1]) if frag_embeds is not None and frag_embeds.ndim == 2 else None
        ),
        "directed_edges": int(edge_index.shape[1]) if edge_index is not None and edge_index.ndim == 2 else 0,
        "LL_directed": edge_counts.get("LL", 0),
        "LP_directed": edge_counts.get("LP", 0),
        "PP_directed": edge_counts.get("PP", 0),
        "DL_directed": edge_counts.get("DL", 0),
        "connections": connection_count,
        "node_pairs_with_two_atom_pairs": sum(
            count == 2 for count in node_pair_counts.values()
        ),
        "connection_distance_min_angstrom": (
            round(float(distance_values.min()), ROUND_DIGITS) if has_distances else None
        ),
        "connection_distance_mean_angstrom": (
            round(float(distance_values.mean()), ROUND_DIGITS) if has_distances else None
        ),
        "connection_distance_max_angstrom": (
            round(float(distance_values.max()), ROUND_DIGITS) if has_distances else None
        ),
    }


def build_validation_rows(data, path: Path) -> list[dict]:
    x = tensor_attr(data, "x")
    num_nodes = int(x.shape[0]) if x is not None and x.ndim == 2 else -1
    edge_index = tensor_attr(data, "edge_index")
    edge_attr = tensor_attr(data, "edge_attr")
    connection_edges = tensor_attr(data, "connection_edge_index")
    connection_atoms = tensor_attr(data, "connection_atom_ids")
    connection_distances = tensor_attr(data, "connection_distance")
    roles = node_roles(data) if num_nodes >= 0 else []
    sample_id = str(getattr(data, "sample_id", path.stem))

    checks = []
    checks.append(("x_shape", x is not None and x.ndim == 2 and x.shape[1] >= len(FEATURE_COLUMNS),
                   None if x is None else str(tuple(x.shape))))
    checks.append(("one_global_node", roles.count("global") == 1,
                   f"global_count={roles.count('global')}"))
    checks.append(("finite_geometry", all(
        finite_tensor(data, name)
        for name in ("x", "pos", "ref_coords", "connection_distance")
    ), "检查 x/pos/ref_coords/connection_distance 中的 NaN 或 Inf"))

    edge_shape_ok = edge_index is not None and edge_index.ndim == 2 and edge_index.shape[0] == 2
    edge_range_ok = edge_shape_ok and (
        edge_index.numel() == 0 or bool(((edge_index >= 0) & (edge_index < num_nodes)).all().item())
    )
    edge_attr_ok = edge_shape_ok and edge_attr is not None and edge_attr.ndim == 2 and (
        edge_attr.shape[0] == edge_index.shape[1] and edge_attr.shape[1] == 4
    )
    checks.append(("edge_index_range", edge_range_ok, "所有边端点应位于节点范围内"))
    checks.append(("edge_attr_shape", edge_attr_ok,
                   None if edge_attr is None else str(tuple(edge_attr.shape))))

    raw_fragment_smiles = getattr(data, "fragment_smiles", None)
    smiles, molecules = fragment_molecules(data)
    fragment_smiles_ok = (
        isinstance(raw_fragment_smiles, (list, tuple))
        and len(raw_fragment_smiles) == num_nodes
        and all(
            (
                roles[node_index] == "global"
                and str(raw_fragment_smiles[node_index]) == "<GLOBAL>"
            )
            or (
                roles[node_index] != "global"
                and molecules[node_index] is not None
            )
            for node_index in range(num_nodes)
        )
    )
    checks.append(("fragment_smiles_canonical_parse", fragment_smiles_ok,
                   None if raw_fragment_smiles is None else f"length={len(smiles)}"))

    connection_fields_present = all(
        value is not None
        for value in (connection_edges, connection_atoms, connection_distances)
    )
    checks.append(("connection_fields_present", connection_fields_present,
                   "三个 connection 字段必须同时存在"))

    connection_edge_shape_ok = (
        connection_fields_present
        and connection_edges.ndim == 2
        and connection_edges.shape[0] == 2
        and connection_edges.dtype == torch.long
    )
    num_connections = int(connection_edges.shape[1]) if connection_edge_shape_ok else -1
    checks.append(("connection_edge_index_shape", connection_edge_shape_ok,
                   None if connection_edges is None else f"{tuple(connection_edges.shape)}, {connection_edges.dtype}"))

    connection_atom_shape_ok = (
        connection_fields_present
        and num_connections >= 0
        and tuple(connection_atoms.shape) == (num_connections, 2)
        and connection_atoms.dtype == torch.long
    )
    checks.append(("connection_atom_ids_shape", connection_atom_shape_ok,
                   None if connection_atoms is None else f"{tuple(connection_atoms.shape)}, {connection_atoms.dtype}"))

    connection_distance_shape_ok = (
        connection_fields_present
        and num_connections >= 0
        and tuple(connection_distances.shape) == (num_connections,)
        and connection_distances.dtype == torch.float32
    )
    checks.append(("connection_distance_shape", connection_distance_shape_ok,
                   None if connection_distances is None else f"{tuple(connection_distances.shape)}, {connection_distances.dtype}"))

    connection_node_range_ok = connection_edge_shape_ok and (
        num_connections == 0
        or bool(((connection_edges >= 0) & (connection_edges < num_nodes)).all().item())
    )
    checks.append(("connection_node_range", connection_node_range_ok,
                   "连接节点编号必须位于真实节点范围内"))

    connection_no_self_loops_ok = connection_edge_shape_ok and (
        num_connections == 0
        or bool((connection_edges[0] != connection_edges[1]).all().item())
    )
    checks.append(("connection_no_self_loops", connection_no_self_loops_ok,
                   "连接不能形成节点自环"))

    connection_node_order_ok = connection_edge_shape_ok and (
        num_connections == 0
        or bool((connection_edges[0] < connection_edges[1]).all().item())
    )
    checks.append(("connection_node_order", connection_node_order_ok,
                   "每个无向节点对只保存 node_a < node_b 的一个方向"))

    connection_ligand_only_ok = connection_node_range_ok and all(
        roles[int(node_a)] == "ligand" and roles[int(node_b)] == "ligand"
        for node_a, node_b in connection_edges.t().tolist()
    )
    checks.append(("connection_ligand_only", connection_ligand_only_ok,
                   "只允许记录 ligand 节点之间的切割连接"))

    connection_atom_range_ok = (
        connection_atom_shape_ok
        and connection_node_range_ok
        and fragment_smiles_ok
        and bool((connection_atoms >= 0).all().item())
    )
    if connection_atom_range_ok:
        for connection_index in range(num_connections):
            node_a = int(connection_edges[0, connection_index])
            node_b = int(connection_edges[1, connection_index])
            atom_a = int(connection_atoms[connection_index, 0])
            atom_b = int(connection_atoms[connection_index, 1])
            if (
                molecules[node_a] is None
                or molecules[node_b] is None
                or atom_a >= molecules[node_a].GetNumAtoms()
                or atom_b >= molecules[node_b].GetNumAtoms()
            ):
                connection_atom_range_ok = False
                break
    checks.append(("connection_canonical_atom_id_range", connection_atom_range_ok,
                   "规范原子编号必须位于各自规范 SMILES 的原子范围内"))

    connection_records = []
    if connection_edge_shape_ok and connection_atom_shape_ok:
        connection_records = [
            (
                int(connection_edges[0, index]),
                int(connection_edges[1, index]),
                int(connection_atoms[index, 0]),
                int(connection_atoms[index, 1]),
            )
            for index in range(num_connections)
        ]
    connection_unique_atom_pairs_ok = (
        connection_edge_shape_ok
        and connection_atom_shape_ok
        and len(set(connection_records)) == len(connection_records)
    )
    checks.append(("connection_unique_atom_pairs", connection_unique_atom_pairs_ok,
                   "同一节点对不能重复保存同一规范原子对"))

    node_pair_counts = {}
    if connection_edge_shape_ok:
        for node_a, node_b in connection_edges.t().tolist():
            node_pair = tuple(sorted((int(node_a), int(node_b))))
            node_pair_counts[node_pair] = node_pair_counts.get(node_pair, 0) + 1
    connection_max_two_pairs_ok = (
        connection_edge_shape_ok
        and all(count <= 2 for count in node_pair_counts.values())
    )
    checks.append(("connection_max_two_atom_pairs_per_node_pair", connection_max_two_pairs_ok,
                   f"最大原子对数={max(node_pair_counts.values(), default=0)}"))

    connection_distance_values_ok = (
        connection_distance_shape_ok
        and bool(torch.isfinite(connection_distances).all().item())
        and bool((connection_distances >= 0).all().item())
    )
    checks.append(("connection_distance_finite_nonnegative", connection_distance_values_ok,
                   "连接原子对距离必须为有限非负数，单位为 Å"))

    return [
        {
            "sample_id": sample_id,
            "file": str(path.relative_to(DATA_DIR)),
            "check": name,
            "passed": bool(passed),
            "detail": detail,
        }
        for name, passed, detail in checks
    ]

## 随机抽样并生成核验表

修改开头的 `N_SAMPLES` 与 `RANDOM_SEED` 可改变抽样。相同文件集合、相同随机种子会得到相同样本。

In [ ]:
normalization_params = load_normalization_params(STATS_PATH)
selected_files = discover_and_sample(DATA_DIR, N_SAMPLES, RANDOM_SEED)
loaded_samples = [(path, trusted_torch_load(path)) for path in selected_files]

print(f"共发现 {len(list(DATA_DIR.rglob('*.pt')))} 个 .pt 文件")
print(f"随机抽取 {len(selected_files)} 个样本（seed={RANDOM_SEED}）:")
for path in selected_files:
    print(f"  - {path.relative_to(DATA_DIR)}")

In [ ]:
summary_rows = []
node_rows = []
edge_rows = []
connection_rows = []
embedding_rows = []
attribute_rows = []
validation_rows = []

for path, data in loaded_samples:
    summary_rows.append(build_summary_row(data, path))
    node_rows.extend(build_node_rows(data, path, normalization_params))
    edge_rows.extend(build_edge_rows(data, path))
    connection_rows.extend(build_connection_rows(data, path))
    embedding_rows.extend(build_embedding_rows(data, path))
    attribute_rows.extend(build_attribute_rows(data, path))
    validation_rows.extend(build_validation_rows(data, path))

tables = {
    "sample_summary": pd.DataFrame(summary_rows),
    "nodes_inverse_normalized": pd.DataFrame(node_rows),
    "unique_undirected_edges": pd.DataFrame(edge_rows),
    "ligand_connections": pd.DataFrame(connection_rows),
    "fragment_embeddings": pd.DataFrame(embedding_rows),
    "attribute_inventory": pd.DataFrame(attribute_rows),
    "validation": pd.DataFrame(validation_rows),
}

tables["sample_summary"]

## 完整表格展示

`nodes_inverse_normalized` 中的特征列已经反向归一化；`pos` 与 `ref_coords` 只是相对配体最小包围球球心进行了中心化，原处理流程没有对它们进行统计归一化，因此这里保持保存值。`ligand_connections` 中的 `connection_distance` 是处理脚本保存的连接原子对距离，单位为 Å；由于 `.pt` 不再保存连接原子的独立坐标，本 notebook 不对该距离进行坐标重算。

In [ ]:
for table_name, dataframe in tables.items():
    display(Markdown(f"### {table_name}（{len(dataframe)} 行）"))
    if dataframe.empty:
        display(Markdown("_无记录_"))
    else:
        display(dataframe)

failed_checks = tables["validation"].loc[~tables["validation"]["passed"]]
if failed_checks.empty:
    display(Markdown("### ✅ 所有结构检查均通过"))
else:
    display(Markdown(f"### ⚠️ 发现 {len(failed_checks)} 项未通过检查"))
    display(failed_checks)

## 可选：导出 CSV

将开头的 `EXPORT_CSV` 改为 `True` 后重新运行，可把每张表导出到 `ChemCut/pyg_sample_review/`。导出的节点表同样使用反向归一化后的特征。

In [ ]:
if EXPORT_CSV:
    EXPORT_DIR.mkdir(parents=True, exist_ok=True)
    sample_tag = "__".join(safe_filename(path.stem) for path in selected_files)
    if len(sample_tag) > 120:
        sample_tag = f"seed_{RANDOM_SEED}_n_{len(selected_files)}"
    for table_name, dataframe in tables.items():
        export_path = EXPORT_DIR / f"{sample_tag}__{table_name}.csv"
        dataframe.to_csv(export_path, index=False, encoding="utf-8-sig")
        print(f"已导出: {export_path}")
else:
    print("EXPORT_CSV=False：本次仅展示，不写出 CSV。")

# 全量 PyG 规模统计与可控清理

本部分会逐个读取 `DATA_DIR` 下的全部 `.pt` 文件，统计节点数和无向边数，并在 notebook 中绘制分布图。每次只保留当前 PyG 对象，避免一次性把整个数据集载入内存。

清理规则只使用节点数和**去重后的无向边数**。`PERFORM_CLEANING = False` 时为纯读取模式；改为 Python 布尔值 `True` 后，超出范围的文件会移动到与数据目录同级的隔离目录，不会直接删除。

In [ ]:
# ================================================================
# 全量清理超参数：请只在这个单元格中调整范围和清理开关
# None 表示该方向不设限制；边界值本身会被保留。
# ================================================================
NODE_COUNT_LIMITS = {
    "min": None,  # 例如 20
    "max": None,  # 例如 500
}

UNDIRECTED_EDGE_COUNT_LIMITS = {
    "min": None,  # 例如 20
    "max": None,  # 例如 5000
}

# False：只读取、统计和绘图；True：将超范围文件移动到隔离目录。
PERFORM_CLEANING = False

QUARANTINE_DIR = DATA_DIR.parent / f"{DATA_DIR.name}_quarantine"

cleaning_config_table = pd.DataFrame([
    {
        "criterion": "node_count",
        "min_inclusive": NODE_COUNT_LIMITS["min"],
        "max_inclusive": NODE_COUNT_LIMITS["max"],
    },
    {
        "criterion": "undirected_edge_count",
        "min_inclusive": UNDIRECTED_EDGE_COUNT_LIMITS["min"],
        "max_inclusive": UNDIRECTED_EDGE_COUNT_LIMITS["max"],
    },
])

display(cleaning_config_table)
print(f"PERFORM_CLEANING = {PERFORM_CLEANING}")
print(f"隔离目录: {QUARANTINE_DIR}")

In [ ]:
import gc
import shutil

import matplotlib.pyplot as plt
from tqdm.auto import tqdm


def validate_count_limits(name: str, limits: dict) -> None:
    if set(limits) != {"min", "max"}:
        raise ValueError(f"{name} 必须且只能包含 min、max")
    lower, upper = limits["min"], limits["max"]
    for key, value in (("min", lower), ("max", upper)):
        if value is not None and (not isinstance(value, int) or isinstance(value, bool) or value < 0):
            raise ValueError(f"{name}.{key} 必须是非负整数或 None")
    if lower is not None and upper is not None and lower > upper:
        raise ValueError(f"{name}.min 不能大于 max")


def count_unique_undirected_edges(edge_index: torch.Tensor) -> int:
    """将 u→v 与 v→u 合并为同一条无向边；自环只计一次。"""
    if not isinstance(edge_index, torch.Tensor):
        raise TypeError("edge_index 不是 Tensor")
    if edge_index.ndim != 2 or edge_index.shape[0] != 2:
        raise ValueError(f"edge_index 形状非法: {tuple(edge_index.shape)}")
    if edge_index.shape[1] == 0:
        return 0
    edge_index = edge_index.detach().cpu().to(torch.long)
    low = torch.minimum(edge_index[0], edge_index[1])
    high = torch.maximum(edge_index[0], edge_index[1])
    undirected_pairs = torch.stack((low, high), dim=1)
    return int(torch.unique(undirected_pairs, dim=0).shape[0])


def scan_all_pyg_files(data_dir: Path) -> pd.DataFrame:
    files = sorted(data_dir.rglob("*.pt"), key=lambda path: path.as_posix())
    if not files:
        raise FileNotFoundError(f"目录中没有 .pt 文件: {data_dir}")

    records = []
    for file_number, path in enumerate(tqdm(files, desc="扫描全部 PyG Data", unit="file"), start=1):
        record = {
            "file": str(path.relative_to(data_dir)),
            "absolute_path": str(path.resolve()),
            "file_size_bytes": path.stat().st_size,
            "load_ok": False,
            "error": "",
            "num_nodes": np.nan,
            "directed_edges": np.nan,
            "undirected_edges": np.nan,
        }
        data = None
        try:
            data = trusted_torch_load(path)
            x = getattr(data, "x", None)
            if isinstance(x, torch.Tensor) and x.ndim >= 1:
                num_nodes = int(x.shape[0])
            else:
                num_nodes = int(data.num_nodes)
            edge_index = getattr(data, "edge_index", None)
            if not isinstance(edge_index, torch.Tensor):
                raise ValueError("缺少有效 edge_index")

            record.update({
                "load_ok": True,
                "num_nodes": num_nodes,
                "directed_edges": int(edge_index.shape[1]),
                "undirected_edges": count_unique_undirected_edges(edge_index),
            })
        except Exception as exc:
            record["error"] = f"{type(exc).__name__}: {exc}"
        finally:
            del data
            if file_number % 500 == 0:
                gc.collect()
        records.append(record)

    gc.collect()
    return pd.DataFrame(records)


def outside_limits(value: int, limits: dict) -> tuple[bool, bool]:
    too_few = limits["min"] is not None and value < limits["min"]
    too_many = limits["max"] is not None and value > limits["max"]
    return bool(too_few), bool(too_many)

In [ ]:
validate_count_limits("NODE_COUNT_LIMITS", NODE_COUNT_LIMITS)
validate_count_limits("UNDIRECTED_EDGE_COUNT_LIMITS", UNDIRECTED_EDGE_COUNT_LIMITS)
if not isinstance(PERFORM_CLEANING, bool):
    raise TypeError("PERFORM_CLEANING 必须是 Python 布尔值 True 或 False")

pyg_dataset_statistics = scan_all_pyg_files(DATA_DIR)
valid_graphs = pyg_dataset_statistics.loc[pyg_dataset_statistics["load_ok"]].copy()
failed_graphs = pyg_dataset_statistics.loc[~pyg_dataset_statistics["load_ok"]].copy()

if valid_graphs.empty:
    raise RuntimeError("没有成功加载任何 PyG Data，停止统计与清理")

for column in ("num_nodes", "directed_edges", "undirected_edges"):
    valid_graphs[column] = valid_graphs[column].astype(np.int64)

dataset_totals = pd.DataFrame([{
    "discovered_files": len(pyg_dataset_statistics),
    "loaded_files": len(valid_graphs),
    "failed_files": len(failed_graphs),
    "total_nodes": int(valid_graphs["num_nodes"].sum()),
    "total_directed_edges": int(valid_graphs["directed_edges"].sum()),
    "total_undirected_edges": int(valid_graphs["undirected_edges"].sum()),
}])

display(Markdown("## 全量读取日志"))
display(dataset_totals)
display(Markdown("### 节点数与无向边数分布统计"))
display(
    valid_graphs[["num_nodes", "undirected_edges"]]
    .describe(percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99])
    .round(2)
)
if not failed_graphs.empty:
    print(f"警告：有 {len(failed_graphs)} 个文件加载失败；加载失败文件不会自动移动。")

In [ ]:
valid_graphs["node_too_few"] = False
valid_graphs["node_too_many"] = False
valid_graphs["edge_too_few"] = False
valid_graphs["edge_too_many"] = False

for row_index, row in valid_graphs.iterrows():
    node_too_few, node_too_many = outside_limits(
        int(row["num_nodes"]), NODE_COUNT_LIMITS
    )
    edge_too_few, edge_too_many = outside_limits(
        int(row["undirected_edges"]), UNDIRECTED_EDGE_COUNT_LIMITS
    )
    valid_graphs.at[row_index, "node_too_few"] = node_too_few
    valid_graphs.at[row_index, "node_too_many"] = node_too_many
    valid_graphs.at[row_index, "edge_too_few"] = edge_too_few
    valid_graphs.at[row_index, "edge_too_many"] = edge_too_many

reason_columns = ["node_too_few", "node_too_many", "edge_too_few", "edge_too_many"]
valid_graphs["should_clean"] = valid_graphs[reason_columns].any(axis=1)
graphs_to_clean = valid_graphs.loc[valid_graphs["should_clean"]].copy()

cleaning_counts = pd.DataFrame({
    "reason": reason_columns,
    "file_count": [int(valid_graphs[column].sum()) for column in reason_columns],
})

print(f"符合保留范围: {int((~valid_graphs['should_clean']).sum())} 个")
print(f"超出清理范围: {len(graphs_to_clean)} 个（按文件去重）")
display(cleaning_counts)

In [ ]:
def add_limit_lines(axis, limits: dict) -> None:
    if limits["min"] is not None:
        axis.axvline(limits["min"], color="#D97706", linestyle="--", linewidth=1.8, label="minimum")
    if limits["max"] is not None:
        axis.axvline(limits["max"], color="#DC2626", linestyle="--", linewidth=1.8, label="maximum")
    if limits["min"] is not None or limits["max"] is not None:
        axis.legend()


histogram_bins = min(80, max(10, int(np.sqrt(len(valid_graphs)))))
figure, axes = plt.subplots(2, 2, figsize=(15, 11))

axes[0, 0].hist(valid_graphs["num_nodes"], bins=histogram_bins, color="#2563EB", alpha=0.82)
add_limit_lines(axes[0, 0], NODE_COUNT_LIMITS)
axes[0, 0].set_title("Node count distribution")
axes[0, 0].set_xlabel("Nodes per PyG Data")
axes[0, 0].set_ylabel("File count")

axes[0, 1].hist(valid_graphs["undirected_edges"], bins=histogram_bins, color="#059669", alpha=0.82)
add_limit_lines(axes[0, 1], UNDIRECTED_EDGE_COUNT_LIMITS)
axes[0, 1].set_title("Undirected edge count distribution")
axes[0, 1].set_xlabel("Unique undirected edges per PyG Data")
axes[0, 1].set_ylabel("File count")

keep_mask = ~valid_graphs["should_clean"]
axes[1, 0].scatter(
    valid_graphs.loc[keep_mask, "num_nodes"],
    valid_graphs.loc[keep_mask, "undirected_edges"],
    s=9, alpha=0.35, color="#64748B", label="keep",
)
if (~keep_mask).any():
    axes[1, 0].scatter(
        valid_graphs.loc[~keep_mask, "num_nodes"],
        valid_graphs.loc[~keep_mask, "undirected_edges"],
        s=13, alpha=0.65, color="#DC2626", label="clean",
    )
axes[1, 0].set_title("Nodes vs undirected edges")
axes[1, 0].set_xlabel("Node count")
axes[1, 0].set_ylabel("Undirected edge count")
axes[1, 0].legend()

axes[1, 1].bar(cleaning_counts["reason"], cleaning_counts["file_count"], color="#7C3AED")
axes[1, 1].set_title("Cleaning reason counts")
axes[1, 1].set_ylabel("File count")
axes[1, 1].tick_params(axis="x", rotation=25)

for axis in axes.flat:
    axis.grid(alpha=0.18)
figure.tight_layout()
plt.show()

In [ ]:
cleaned_file_count = 0
cleaning_failure_count = 0

if not PERFORM_CLEANING:
    print("纯读取模式：PERFORM_CLEANING=False，本次实际清理 0 个文件。")
    print(f"若改为 True，按当前范围将清理 {len(graphs_to_clean)} 个文件。")
else:
    data_root = DATA_DIR.resolve()
    quarantine_root = QUARANTINE_DIR.resolve()
    if quarantine_root == data_root or data_root in quarantine_root.parents:
        raise ValueError("隔离目录不能等于数据目录，也不能位于数据目录内部")
    quarantine_root.mkdir(parents=True, exist_ok=True)

    for row in tqdm(graphs_to_clean.itertuples(index=False), total=len(graphs_to_clean), desc="移动超范围 PyG Data", unit="file"):
        source = Path(row.absolute_path).resolve()
        try:
            relative_path = source.relative_to(data_root)
            target = (quarantine_root / relative_path).resolve()
            target.relative_to(quarantine_root)
            if not source.is_file():
                raise FileNotFoundError(f"源文件不存在: {source}")
            if target.exists():
                raise FileExistsError(f"隔离目录中已存在同名文件: {target}")
            target.parent.mkdir(parents=True, exist_ok=True)
            shutil.move(str(source), str(target))
            cleaned_file_count += 1
        except Exception as exc:
            cleaning_failure_count += 1
            tqdm.write(f"[清理失败] {source}: {type(exc).__name__}: {exc}")

    print(f"清理完成：成功移动 {cleaned_file_count} 个文件。")
    print(f"清理失败：{cleaning_failure_count} 个文件。")
    print(f"隔离目录：{quarantine_root}")